# Creating Convokit Corpus element
according to https://github.com/CornellNLP/ConvoKit/blob/master/examples/converting_movie_corpus.ipynb

In [2]:
from convokit import Corpus, Speaker, Utterance
import pandas as pd
import tqdm
import ast

In [3]:
media_sum_path = "data/MediaSum/news_dialogue.json"
media_sum_json = pd.read_json(media_sum_path)

In [4]:
media_sum_json

,id,program,date,url,title,summary,utt,speaker
0,NPR-1,News & Notes,2007-11-28,https://www.npr.org/templates/story/story.php?...,Black Actors Give Bible Star Appeal,"More than 400 black actors, artists and minist...","[Now, moving on, Forest Whitaker as Moses, Tis...","[FARAI CHIDEYA, host, FARAI CHIDEYA, host, Mr...."
1,NPR-2,Weekend Edition Sunday,2016-10-23,https://www.npr.org/2016/10/23/499042298/young...,"Young, First-Time Voters Share Views On Electi...",NPR's Rachel Martin speaks with young voters w...,[You have heard it again and again - this is a...,"[RACHEL MARTIN, HOST, ASHANTI MARTINEZ, LAUREN..."
2,NPR-3,News & Notes,2007-11-30,https://www.npr.org/templates/story/story.php?...,Snapshots: On Solid Ground,"In this week's snapshot, actor and playwright ...","[I came close to running out of luck, when I a...","[Mr. JEFF OBAFEMI CARR (Actor, Playwright), CH..."
3,NPR-4,News & Notes,2007-11-30,https://www.npr.org/templates/story/story.php?...,"Washington, D.C. Facing HIV/AIDS Epidemic",A new study says one in 50 people in the natio...,"[This is NEWS & NOTES. I'm Farai Chideya., In ...","[FARAI CHIDEYA, host, FARAI CHIDEYA, host, Dr...."
4,NPR-5,News & Notes,2007-11-30,https://www.npr.org/templates/story/story.php?...,Coping When AIDS Hits Your Family: Part II,When a family member is diagnosed with HIV/AID...,"[I'm Farai Chideya and this is NEWS & NOTES., ...","[FARAI CHIDEYA, host, FARAI CHIDEYA, host, FAR..."
...,...,...,...,...,...,...,...,...
463591,CNN-414237,CNN NEWSROOM,2020-10-25,http://transcripts.cnn.com/TRANSCRIPTS/2010/25...,NaN,"U.S. Officials: Russia, Iran Have Stolen Voter...",[Welcome back to our viewers in the United Sta...,"[BRUNHUBER, NATASHA CHEN, CNN CORRESPONDENT, W..."
463592,CNN-414238,CNN NEWSROOM,2020-10-25,http://transcripts.cnn.com/TRANSCRIPTS/2010/25...,NaN,Nigerian Police Force Mobilize To Quell Worst ...,"[In Nigeria, chaotic scenes of looting and des...","[BRUNHUBER, BRUNHUBER (voice-over), BRUNHUBER ..."
463593,CNN-414239,CNN NEWSROOM,2020-10-25,http://transcripts.cnn.com/TRANSCRIPTS/2010/25...,NaN,COVID-19 Triggers Rise In Asian American Unemp...,[Officials in the U.S. are worried about wides...,"[BRUNHUBER, AMARA WALKER, CNN ANCHOR (voice-ov..."
463594,CNN-414240,STATE OF THE UNION,2020-10-25,http://transcripts.cnn.com/TRANSCRIPTS/2010/25...,NaN,COVID-19 Outbreak Hits Vice President Pence's ...,[Dark winter? U.S. COVID cases hit a new daily...,"[JAKE TAPPER, CNN HOST (voice-over), DONALD TR..."


## 1. Create speakers

**Note**: In the speaker list, authors sometimes have non-unique identifiers (e.g., ‘STEVE PROFFITT’, ‘PROFFITT’ or ‘S. PROFFITT’ refer to the same speaker). See example below. Currently I **do not** address this. I will count each unique identifier as a different speaker. Plus, I will count an identifier that is the same in one conversation as in another as the same speaker in another conversation. This might be incorrect for cases like below with 'UNIDENTIFIED MALE' or 'UNIDENTIFIED FEMALE', but I will not address this for now.

In [5]:
media_sum_json["speaker"][300000]

['CUOMO',
 'ED LAVANDERA, CNN CORRESPONDENT',
 'LAVANDERA (voice-over)',
 'ERIC HOLDER, U.S. ATTORNEY GENERAL',
 'LAVANDERA',
 'UNIDENTIFIED FEMALE',
 'UNIDENTIFIED MALE',
 'UNIDENTIFIED MALE',
 'LAVANDERA',
 'HOLDER',
 'LAVANDERA',
 'LAVANDERA',
 'PEREIRA',
 'PASTOR ROBERT WHITE, PEACE OF MIND CHURCH OF HAPPINESS',
 'PEREIRA',
 'MO IVORY, ATTORNEY/RADIO PERSONALITY',
 'PEREIRA',
 'IVORY',
 'PEREIRA',
 'IVORY',
 'PEREIRA',
 'WHITE',
 'PEREIRA',
 'WHITE',
 'PEREIRA',
 'WHITE',
 'PEREIRA',
 'WHITE',
 'PEREIRA',
 'IVORY',
 'WHITE',
 'IVORY',
 'PEREIRA',
 'IVORY',
 'PEREIRA',
 'WHITE',
 'PEREIRA',
 'WHITE',
 'PEREIRA',
 'CUOMO',
 'BERMAN']

I use the incorrect **assumption that each element in the speaker list is a string that is the only unique string for this speaker across the whole dataset**.

In [6]:
# get all speakers from the speaker column
speakers = media_sum_json['speaker']
unique_speakers = sorted(set(name for sublist in speakers for name in sublist))

I create a speaker object that only includes the speaker name as information and identifier.

In [7]:
corpus_speakers = {speaker_name: Speaker(id = speaker_name, meta ={'name': speaker_name}) for speaker_name in unique_speakers}

In [8]:
corpus_speakers['LAVANDERA']

Speaker({'obj_type': 'speaker', 'vectors': [], 'owner': None, 'id': 'LAVANDERA', 'temp_backend': {}, 'meta': {'name': 'LAVANDERA'}})

In [9]:
corpus_speakers['ED LAVANDERA, CNN CORRESPONDENT']

Speaker({'obj_type': 'speaker', 'vectors': [], 'owner': None, 'id': 'ED LAVANDERA, CNN CORRESPONDENT', 'temp_backend': {}, 'meta': {'name': 'ED LAVANDERA, CNN CORRESPONDENT'}})

## 2. Creating utterance objects

In [10]:
type(media_sum_json['utt'][0])

list

In [11]:
utterance_corpus = {}
conversation_meta = {}

count = 0
# iterate over each row in the dataframe
for index, row in tqdm.tqdm(media_sum_json.iterrows(), total=media_sum_json.shape[0]):
    # get the conversation id
    conversation_id = row['id']
    program = row['program']
    date = row['date']
    summary = row['summary']
    url = row['url']
    title = row['title']

    conversation_meta[conversation_id] = {
        'program': program,
        'date': date,
        'summary': summary,
        'url': url,
        'title': title,
        'broadcaster': conversation_id.split('-')[0],  # should be either NPR or CNN
    }

    # get utterance information
    utterance_list = row['utt']
    speaker_list = row['speaker']

    for i, utt in enumerate(utterance_list):
        # create a unique identifier for the utterance as in https://aclanthology.org/2024.emnlp-main.52.pdf
        #   i.e., from the code base ID of the form 'CNN-67148-13' where 'CNN-67148' is the identifier as used in MediaSum and 13 is the index of the utterance in the original utterance list
        utterance_id = f"{conversation_id}-{i}"
        utt_speaker = corpus_speakers[speaker_list[i]]
        utt_text = utt
        reply_to = None if i == 0 else f"{conversation_id}-{i-1}"  # reply_to is None for the first utterance in the conversation
        # timestamp is not provided

        utterance_corpus[utterance_id] = Utterance(
            id=utterance_id,
            speaker=utt_speaker,
            conversation_id=conversation_id,
            reply_to=reply_to,
            text=utt_text,
        )

print(f"Total number of utterances: {len(utterance_corpus)}")

100%|█████████████████████████████████| 463596/463596 [00:57<00:00, 8064.32it/s]

Total number of utterances: 13919244


In [12]:
# example utterance
utterance_corpus['CNN-67148-13']

Utterance({'obj_type': 'utterance', 'vectors': [], 'speaker_': Speaker({'obj_type': 'speaker', 'vectors': [], 'owner': None, 'id': 'CLARK', 'temp_backend': {}, 'meta': {'name': 'CLARK'}}), 'owner': None, 'id': 'CNN-67148-13', 'temp_backend': {'speaker_id': 'CLARK', 'conversation_id': 'CNN-67148', 'reply_to': 'CNN-67148-12', 'timestamp': None, 'text': "Well, I don't think -- as far as I know, we're not paying anything to Saudi Arabia, for example, right now. In fact, they're still buying weapons. They are having economic difficulties, but they do have oil. But the other countries in the region are in one way or another in financial trouble, and have been for a long time. They've been sustained on a diet of expectations of economic growth, funded by taking short and long term loans that come from commercial banks, sometimes guaranteed by governments. And then they have to repay these loans. And repaying these loans consumes their foreign exchange earnings from their exports and from remi

In [13]:
utterance_corpus["NPR-1-0"]

Utterance({'obj_type': 'utterance', 'vectors': [], 'speaker_': Speaker({'obj_type': 'speaker', 'vectors': [], 'owner': None, 'id': 'FARAI CHIDEYA, host', 'temp_backend': {}, 'meta': {'name': 'FARAI CHIDEYA, host'}}), 'owner': None, 'id': 'NPR-1-0', 'temp_backend': {'speaker_id': 'FARAI CHIDEYA, host', 'conversation_id': 'NPR-1', 'reply_to': None, 'timestamp': None, 'text': 'Now, moving on, Forest Whitaker as Moses, Tisha Campbell Martin as Mary Magdalene - well, that\'s all in "The Bible Experience." A New Testament edition was released in 2006. This edition is billed as "The Complete Bible." It doesn\'t have one person reading the gospels. It features nearly 400 African-American artists, actors and ministers, plus sound effects.'}, 'meta': {}})

## 3. Creating corpus from list of utterances

In [14]:
utterance_list = utterance_corpus.values()

In [15]:
media_sum_corpus = Corpus(utterances=utterance_list)

In [16]:
print("number of conversations in the dataset = {}".format(len(media_sum_corpus.get_conversation_ids())))

number of conversations in the dataset = 463596


In [17]:
convo_ids = media_sum_corpus.get_conversation_ids()
for i, convo_idx in enumerate(convo_ids[0:5]):
    print("sample conversation {}:".format(i))
    print(media_sum_corpus.get_conversation(convo_idx).get_utterance_ids())

sample conversation 0:
['NPR-1-0', 'NPR-1-1', 'NPR-1-2', 'NPR-1-3', 'NPR-1-4', 'NPR-1-5', 'NPR-1-6', 'NPR-1-7', 'NPR-1-8', 'NPR-1-9', 'NPR-1-10', 'NPR-1-11', 'NPR-1-12', 'NPR-1-13', 'NPR-1-14', 'NPR-1-15', 'NPR-1-16', 'NPR-1-17', 'NPR-1-18', 'NPR-1-19', 'NPR-1-20', 'NPR-1-21', 'NPR-1-22', 'NPR-1-23', 'NPR-1-24', 'NPR-1-25', 'NPR-1-26', 'NPR-1-27', 'NPR-1-28', 'NPR-1-29', 'NPR-1-30', 'NPR-1-31', 'NPR-1-32', 'NPR-1-33', 'NPR-1-34', 'NPR-1-35', 'NPR-1-36', 'NPR-1-37', 'NPR-1-38', 'NPR-1-39', 'NPR-1-40', 'NPR-1-41', 'NPR-1-42', 'NPR-1-43', 'NPR-1-44', 'NPR-1-45', 'NPR-1-46', 'NPR-1-47']
sample conversation 1:
['NPR-2-0', 'NPR-2-1', 'NPR-2-2', 'NPR-2-3', 'NPR-2-4', 'NPR-2-5', 'NPR-2-6', 'NPR-2-7', 'NPR-2-8', 'NPR-2-9', 'NPR-2-10', 'NPR-2-11', 'NPR-2-12', 'NPR-2-13', 'NPR-2-14', 'NPR-2-15', 'NPR-2-16', 'NPR-2-17', 'NPR-2-18', 'NPR-2-19', 'NPR-2-20', 'NPR-2-21', 'NPR-2-22', 'NPR-2-23', 'NPR-2-24', 'NPR-2-25', 'NPR-2-26', 'NPR-2-27', 'NPR-2-28', 'NPR-2-29', 'NPR-2-30', 'NPR-2-31', 'NPR-2-32', 

## 4. Updating Conversation and Corpus level metadata

In [18]:
for convo in media_sum_corpus.iter_conversations():
    # get the conversation id by checking from utterance info
    convo_id = convo.get_id()

    # update meta with additional conversation information
    convo.meta.update(conversation_meta[convo_id])

In [19]:
media_sum_corpus.get_conversation("CNN-67148").meta

ConvoKitMeta({'program': 'CNN SATURDAY NIGHT', 'date': '2003-2-22', 'summary': 'How Much Will War With Iraq Cost?', 'url': 'http://transcripts.cnn.com/TRANSCRIPTS/0302/22/stn.02.html', 'title': nan, 'broadcaster': 'CNN'})

In [20]:
media_sum_corpus.get_conversation("NPR-1").meta

ConvoKitMeta({'program': 'News & Notes', 'date': '2007-11-28', 'summary': 'More than 400 black actors, artists and ministers are bringing the Gospel to life in the audio book, The Bible Experience:The Complete Bible. Farai Chideya talks with producer Kyle Bowser and actress Wendy Raquel Robinson, who lends her voice to the project.', 'url': 'https://www.npr.org/templates/story/story.php?storyId=16697288', 'title': 'Black Actors Give Bible Star Appeal', 'broadcaster': 'NPR'})

In [21]:
media_sum_corpus.meta['name'] = 'MediaSum Corpus'

## 5. Adding Paraphrase annotations

Annotations are saved as lists which correspond to the text with utt.text.split() calls.

In [22]:
# load annotations from huggingface dataset
from datasets import load_dataset
dataset = load_dataset("AnnaWegmann/Paraphrases-in-Interviews")

In [23]:
split_names = list(dataset.keys())
dataframes = [dataset[split].to_pandas() for split in split_names]
df = pd.concat(dataframes, ignore_index=True)  # if you just need one split: dataset['train'].to_pandas()

In [24]:
utterance = media_sum_corpus.get_utterance("CNN-177596-7")

In [25]:
utterance.text

'This is not good.'

In [26]:
unique_annotators = set(df['Annotator'])
len(unique_annotators)

112

In [27]:
# get all unique QIDs in the dataset
unique_qids = set(df['QID'].unique())

In [28]:
# go over the unique QIDs
for q_id in tqdm.tqdm(unique_qids):
    group = df[df['QID'] == q_id]
    # Compute total votes and paraphrase votes
    total_votes = len(group)
    paraphrase_votes = group['Is Paraphrase'].astype(int).sum()

    meta_info = {
        'total_votes': int(total_votes),
        'paraphrase_votes': int(paraphrase_votes),
        'paraphrase_ratio': float(paraphrase_votes / total_votes if total_votes > 0 else 0)
    }

    # Process Guest Highlights
    guest_highlights_list = group['Guest Highlights'].apply(ast.literal_eval).tolist()
    guest_highlights_sums = [sum(x) for x in zip(*guest_highlights_list)]

    # Process Host Highlights
    host_highlights_list = group['Host Highlights'].apply(ast.literal_eval).tolist()
    host_highlights_sums = [sum(x) for x in zip(*host_highlights_list)]

    cur_utt = media_sum_corpus.get_utterance(q_id)
    utt_number = int(q_id.split("-")[2])
    # guest_speaker = cur_utt.speaker.id
    cur_id = 0
    while cur_id < len(guest_highlights_sums):
        cur_utt_text_len = len(cur_utt.text.split())
        meta_info['Guest Highlights'] = guest_highlights_sums[cur_id:cur_utt_text_len]
        # meta_info['Guest Words'] = cur_utt.text.split()
        meta_info['is_host'] = False
        # print(meta_info)
        for key in meta_info.keys():
            cur_utt.add_meta(key, meta_info[key])
        for index, row in group.iterrows():
            cur_utt.add_meta(row['Annotator'], ast.literal_eval(row['Guest Highlights'])[cur_id:cur_utt_text_len])
        del meta_info['Guest Highlights']
        # del meta_info['Guest Words']
        del meta_info['is_host']
        utt_number+=1
        cur_utt = media_sum_corpus.get_utterance(f"{cur_utt.conversation_id}-{utt_number}")
        cur_id += cur_utt_text_len
    # host_speaker = cur_utt.speaker.id
    cur_id = 0
    while cur_id < len(host_highlights_sums):
        cur_utt_text_len = len(cur_utt.text.split())
        meta_info['Host Highlights'] = host_highlights_sums[cur_id:cur_utt_text_len]
        # meta_info['Host Words'] = cur_utt.text.split()
        meta_info['is_host'] = True
        for key in meta_info.keys():
            cur_utt.add_meta(key, meta_info[key])
        for index, row in group.iterrows():
            cur_utt.add_meta(row['Annotator'], ast.literal_eval(row['Host Highlights'])[cur_id:cur_utt_text_len])
        del meta_info['Host Highlights']
        # del meta_info['Host Words']
        del meta_info['is_host']
        utt_number+=1
        cur_utt = media_sum_corpus.get_utterance(f"{cur_utt.conversation_id}-{utt_number}")
        cur_id += cur_utt_text_len

100%|████████████████████████████████████████| 600/600 [00:01<00:00, 322.53it/s]


In [29]:
utterance = media_sum_corpus.get_utterance("CNN-177596-7")
utterance.meta, utterance.text

(ConvoKitMeta({'total_votes': 20, 'paraphrase_votes': 10, 'paraphrase_ratio': 0.5, 'Guest Highlights': [10, 9, 9, 9], 'is_host': False, 'PROLIFIC_1': [0, 0, 0, 0], 'PROLIFIC_2': [1, 1, 1, 1], 'PROLIFIC_3': [0, 0, 0, 0], 'PROLIFIC_4': [0, 0, 0, 0], 'PROLIFIC_5': [0, 0, 0, 0], 'PROLIFIC_6': [1, 1, 1, 1], 'PROLIFIC_7': [1, 0, 0, 0], 'PROLIFIC_8': [1, 1, 1, 1], 'PROLIFIC_9': [0, 0, 0, 0], 'PROLIFIC_10': [0, 0, 0, 0], 'PROLIFIC_11': [1, 1, 1, 1], 'PROLIFIC_12': [0, 0, 0, 0], 'PROLIFIC_13': [1, 1, 1, 1], 'PROLIFIC_14': [0, 0, 0, 0], 'PROLIFIC_15': [0, 0, 0, 0], 'PROLIFIC_16': [1, 1, 1, 1], 'PROLIFIC_17': [0, 0, 0, 0], 'PROLIFIC_18': [1, 1, 1, 1], 'PROLIFIC_19': [1, 1, 1, 1], 'PROLIFIC_20': [1, 1, 1, 1]}),
 'This is not good.')

In [30]:
utterance = media_sum_corpus.get_utterance("CNN-177596-8")
utterance.meta, utterance.text

(ConvoKitMeta({'total_votes': 20, 'paraphrase_votes': 10, 'paraphrase_ratio': 0.5, 'Host Highlights': [9, 8, 9, 9, 9, 9, 9, 7, 7, 7, 1], 'is_host': True, 'PROLIFIC_1': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'PROLIFIC_2': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0], 'PROLIFIC_3': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'PROLIFIC_4': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'PROLIFIC_5': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'PROLIFIC_6': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0], 'PROLIFIC_7': [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'PROLIFIC_8': [1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0], 'PROLIFIC_9': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'PROLIFIC_10': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'PROLIFIC_11': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'PROLIFIC_12': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'PROLIFIC_13': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0], 'PROLIFIC_14': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'PROLIFIC_15': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'PROLIFIC_16': [0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0], 'PROLIFIC_17': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'PROL

In [31]:
utterance = media_sum_corpus.get_utterance("CNN-80522-7")
utterance.meta, utterance.text

(ConvoKitMeta({'total_votes': 3, 'paraphrase_votes': 3, 'paraphrase_ratio': 1.0, 'Guest Highlights': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 1, 0], 'is_host': False, 'PROLIFIC_36': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0], 'PROLIFIC_40': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0], 'PROLIFIC_53': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [33]:
utterance = media_sum_corpus.get_utterance("CNN-80522-8")
utterance.meta, utterance.text

(ConvoKitMeta({'total_votes': 3, 'paraphrase_votes': 3, 'paraphrase_ratio': 1.0, 'Host Highlights': [0, 3, 3, 3, 3, 3, 3, 3, 3], 'is_host': True, 'PROLIFIC_36': [0, 1, 1, 1, 1, 1, 1, 1, 1], 'PROLIFIC_40': [0, 1, 1, 1, 1, 1, 1, 1, 1], 'PROLIFIC_53': [0, 1, 1, 1, 1, 1, 1, 1, 1]}),
 "Yes, you've been doing this for a while now.")

## 6. Saving created datsets

In [34]:
media_sum_corpus.dump("mediasum-corpus")